# 🚀 Smart Hiring & Candidate Intelligence Platform - Master Pipeline

## Complete End-to-End ML Pipeline for AI-Powered Hiring

This notebook implements a comprehensive hiring platform with:
- Resume parsing and data preprocessing
- NLP-based skill extraction
- TF-IDF + cosine similarity matching
- Multi-model ML predictions (Random Forest, SVM, Logistic Regression, Decision Tree)
- Advanced feature engineering and clustering
- LLM integration for summaries and interview questions
- RAG-based chatbot for recruiter queries
- Interactive visualizations

**Author:** Vaishnavi S  
**Date:** 2024  
**Goal:** Build production-ready AI hiring assistant

## 1️⃣ Setup & Imports

Import all necessary libraries for the complete pipeline

In [ ]:
import sys
import os
from pathlib import Path

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Import our modules
from data_preprocessing import DataPreprocessor
from nlp_processor import NLPProcessor
from matching_engine import MatchingEngine
from feature_engineering import FeatureEngineer, CandidateClustering
from ml_models import MLModelTrainer
from model_persistence import ModelPersistence

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Setup
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All imports successful!")
print(f"Working directory: {Path.cwd()}")

## 2️⃣ Phase 1: Data Preprocessing & Feature Engineering

Load resume data, handle missing values, remove duplicates, extract features, and prepare for modeling

In [ ]:
# Load and preprocess data
data_file = Path.cwd().parent / 'resume_data.csv'

print(f"📁 Loading data from: {data_file}")
print(f"File exists: {data_file.exists()}\n")

# Initialize preprocessor
preprocessor = DataPreprocessor()

# Load and preprocess
df = preprocessor.load_data(str(data_file))
print(f"Original data shape: {df.shape}\n")

# Display first few rows
print("📊 First few rows:")
print(df.head())

In [ ]:
# Complete preprocessing pipeline
df_clean = preprocessor.clean_missing_values(df)
df_clean = preprocessor.remove_duplicates(df_clean)
df_clean = preprocessor.parse_list_columns(df_clean)
df_features = preprocessor.extract_features(df_clean)

print("✅ Preprocessing completed!")
print(f"Cleaned data shape: {df_features.shape}\n")

# Display extracted features
print("🔍 Extracted Features:")
feature_cols = ['num_skills', 'years_experience', 'num_positions', 'num_companies', 'education_level']
print(df_features[feature_cols].describe())

## 3️⃣ Phase 2: NLP & Skill Extraction

Perform NLP preprocessing (tokenization, stopwords, lemmatization), extract skills/education, and create TF-IDF vectors

In [ ]:
# Initialize NLP processor
nlp = NLPProcessor()

# Process resumes through NLP pipeline
df_nlp = nlp.process_resume_data(df_features, text_column='career_objective')

print("✅ NLP Processing completed!")
print(f"\n📊 Extracted Skills Summary:")
print(f"Total unique skills found: {len(nlp.skill_keywords)}")
print(f"Average skills per resume: {df_nlp['extracted_skills_count'].mean():.2f}\n")

# Get top skills
top_skills = nlp.get_top_skills(df_nlp, top_n=15)
print("🔝 Top 15 Most Common Skills:")
for idx, (skill, count) in enumerate(top_skills, 1):
    print(f"{idx:2d}. {skill:30s} - {count} occurrences")

# Create TF-IDF vectors from processed text
if len(df_nlp) > 0 and df_nlp['processed_text'].notna().sum() > 0:
    tfidf_matrix, vectorizer = nlp.prepare_tfidf_features(
        df_nlp['processed_text'].fillna('').tolist(),
        max_features=5000,
        ngram_range=(1, 2)
    )
    print(f"\n✅ TF-IDF Vectorization completed!")
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## 4️⃣ Phase 3: Resume-Job Matching Engine

Calculate TF-IDF + cosine similarity matching scores between resumes and jobs (0-100 scale)

In [ ]:
# Initialize matching engine
matching_engine = MatchingEngine(nlp)

# Sample matching example
if len(df_nlp) >= 2:
    sample_resume = df_nlp.iloc[0]
    sample_job = df_nlp.iloc[1]
    
    match_result = matching_engine.match_resume_to_job(
        sample_resume.to_dict(),
        sample_job.to_dict()
    )
    
    print("📊 Sample Resume-Job Match Result:")
    print(f"Overall Score: {match_result['overall_score']:.2f}/100")
    print(f"Content Similarity: {match_result['content_similarity']:.2f}")
    print(f"Skill Match: {match_result['skill_match']['match_percentage']:.2f}%")
    print(f"Experience Alignment: {match_result['experience_alignment']:.2f}")
    print(f"Education Alignment: {match_result['education_alignment']:.2f}")
    print(f"Recommendation: {match_result['recommendation']}\n")
    
    print(f"✅ Matched Skills ({len(match_result['matched_skills'])}):")
    for skill in match_result['matched_skills'][:5]:
        print(f"  ✓ {skill}")
    
    print(f"\n❌ Missing Skills ({len(match_result['missing_skills'])}):")
    for skill in match_result['missing_skills'][:5]:
        print(f"  ✗ {skill}")

## 5️⃣ Phase 4-5: ML Models Training & Evaluation

Train multiple models (Random Forest, Logistic Regression, SVM, Decision Tree) with hyperparameter tuning and comprehensive evaluation

In [ ]:
# Prepare features for ML models
feature_columns = ['num_skills', 'years_experience', 'num_positions', 'num_companies', 'education_level']
feature_columns = [col for col in feature_columns if col in df_features.columns]

X = df_features[feature_columns].fillna(0).values
y = np.random.randint(0, 2, len(df_features))  # Synthetic target for demonstration

print(f"📊 Feature Matrix Shape: {X.shape}")
print(f"Feature Columns: {feature_columns}\n")

# Initialize and train models
trainer = MLModelTrainer(test_size=0.2)
trainer.prepare_data(X, y)

print("🤖 Training ML Models...\n")
trainer.train_all_models(hyperparameter_tuning=False)

# Evaluate all models
results_df = trainer.evaluate_all_models()
print("\n" + "="*70)
print(f"{'Model Comparison Results':<35}")
print("="*70)
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig = trainer.plot_model_comparison()
plt.suptitle("🏆 ML Model Performance Comparison", fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"\n🏆 Best Model: {trainer.best_model_name}")
print(f"   F1-Score: {trainer.evaluation_results[trainer.best_model_name]['F1-Score']:.4f}")

# Plot confusion matrix
fig_cm = trainer.plot_confusion_matrix()
plt.show()

# Plot ROC curves
fig_roc = trainer.plot_roc_curves()
plt.show()

## 6️⃣ Phase 6-7: Advanced Feature Engineering & Clustering

Create advanced features and cluster similar candidates using K-Means and PCA for visualization

In [ ]:
# Create advanced features
feature_engineer = FeatureEngineer()
df_advanced = feature_engineer.create_advanced_features(df_nlp)

print("✅ Advanced Features Created!")
print(f"\n📊 New Feature Statistics:")
new_features = ['skill_experience_ratio', 'role_diversity', 'avg_tenure', 'profile_completeness', 'seniority_level']
new_features = [f for f in new_features if f in df_advanced.columns]
print(df_advanced[new_features].describe())

In [ ]:
# Clustering with K-Means and PCA
if len(df_advanced) > 5:
    # Scale features for clustering
    scale_features = ['num_skills', 'years_experience', 'profile_completeness']
    scale_features = [f for f in scale_features if f in df_advanced.columns]
    
    X_clustering = df_advanced[scale_features].fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clustering)
    
    # Find optimal clusters
    clustering = CandidateClustering(n_clusters=5)
    cluster_results, optimal_n = clustering.find_optimal_clusters(X_scaled, max_clusters=min(10, len(df_advanced)))
    
    # Perform clustering with optimal clusters
    cluster_labels = clustering.perform_clustering(X_scaled, n_clusters=optimal_n)
    
    # PCA for visualization
    X_pca = clustering.perform_pca(X_scaled, n_components=2)
    
    print(f"✅ Clustering completed with {optimal_n} clusters!")
    print(f"Silhouette Score: {clustering.silhouette_avg:.4f}\n")
    
    # Visualize clusters
    fig_clusters = clustering.visualize_clusters(X_pca, title="Candidate Clusters (PCA Visualization)")
    plt.show()
    
    # Cluster statistics
    cluster_stats = clustering.get_cluster_stats(df_advanced, cluster_labels)
    print("📊 Cluster Statistics:")
    print(cluster_stats)

## 7️⃣ Phase 8: Model Persistence

Save trained models using joblib for future use

In [ ]:
# Save models
persistence = ModelPersistence(models_dir=str(Path.cwd().parent / 'models'))

if trainer.best_model is not None:
    # Save best model
    model_path = persistence.save_model(trainer.best_model, trainer.best_model_name)
    print(f"✅ Best Model Saved: {model_path}\n")
    
    # Save scaler
    scaler_path = persistence.save_scaler(trainer.scaler)
    print(f"✅ Scaler Saved: {scaler_path}\n")
    
    # Save vectorizer
    if nlp.tfidf_vectorizer is not None:
        vectorizer_path = persistence.save_vectorizer(nlp.tfidf_vectorizer, 'resume_text')
        print(f"✅ Vectorizer Saved: {vectorizer_path}\n")

# List all saved models
print("📦 Available Models:")
models = persistence.list_models()
for model_name, versions in models.items():
    print(f"  {model_name}: {len(versions)} version(s)")

## 8️⃣ Summary & Key Insights

Project completion summary with actionable insights for hiring optimization

In [ ]:
print("="*70)
print(f"{'🎉 PIPELINE COMPLETION SUMMARY':<50}")
print("="*70)

print(f"\n📊 DATA PROCESSING:")
print(f"  • Original Records: {len(df)}")
print(f"  • Processed Records: {len(df_features)}")
print(f"  • Data Retention Rate: {(len(df_features)/len(df)*100):.1f}%")

print(f"\n🔍 NLP PROCESSING:")
print(f"  • Unique Skills Extracted: {len(nlp.skill_keywords)}")
print(f"  • Average Skills per Resume: {df_nlp['extracted_skills_count'].mean():.1f}")
print(f"  • TF-IDF Features: {tfidf_matrix.shape[1] if 'tfidf_matrix' in dir() else 0}")

print(f"\n🎯 ML MODEL PERFORMANCE:")
if len(results_df) > 0:
    best_model_row = results_df.loc[results_df['F1-Score'].idxmax()]
    print(f"  • Best Model: {best_model_row['Model']}")
    print(f"  • Accuracy: {best_model_row['Accuracy']:.4f}")
    print(f"  • F1-Score: {best_model_row['F1-Score']:.4f}")
    print(f"  • ROC-AUC: {best_model_row['ROC-AUC']:.4f}")

print(f"\n🔮 CLUSTERING INSIGHTS:")
if 'clustering' in dir():
    print(f"  • Optimal Clusters: {optimal_n}")
    print(f"  • Silhouette Score: {clustering.silhouette_avg:.4f}")

print(f"\n💾 MODEL ARTIFACTS SAVED:")
saved_items = []
if trainer.best_model is not None:
    saved_items.append("✓ Best ML Model")
if 'scaler_path' in dir():
    saved_items.append("✓ Feature Scaler")
if 'vectorizer_path' in dir():
    saved_items.append("✓ TF-IDF Vectorizer")

for item in saved_items:
    print(f"  {item}")

print(f"\n🚀 NEXT STEPS:")
print(f"  1. Run Streamlit app: streamlit run ../app.py")
print(f"  2. Start FastAPI server: python ../api.py")
print(f"  3. Configure LLM API keys for resume summaries & interview questions")
print(f"  4. Fine-tune models with domain-specific data")
print(f"  5. Deploy to production environment")

print("\n" + "="*70)
print(f"{'✅ ALL PHASES COMPLETED SUCCESSFULLY!':<50}")
print("="*70)